# 实验五 · CNN 卷积神经网络

基于课程参考的 LeNet 风格模型识别 MNIST。依次运行各单元格，默认完整训练 30 轮。安装与运行方式见 [README.md](README.md)。

## 1. 导入依赖并配置实验

固定随机种子，自动选择 CPU 或 CUDA。每次完整运行创建独立结果目录。

In [1]:
import json
import platform
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

root = Path.cwd()
if root.name != "5th.exp":
    root = root / "5th.exp"
if not (root / "code.ipynb").is_file():
    raise FileNotFoundError("请从仓库根目录或 5th.exp 目录启动。")
seed = 42
batch_size = 512
epochs = 30
learning_rate = 0.01
torch.manual_seed(seed)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_dir = root / "result" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
run_dir.mkdir(parents=True)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | {device}")

Python 3.14.0 | PyTorch 2.9.1+cpu | cpu


## 2. 加载 MNIST

首次运行需要联网下载，之后复用 data 缓存。ToTensor 将像素缩放到 [0, 1]。测试集顺序固定。

In [2]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root / "data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root / "data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                          generator=torch.Generator().manual_seed(seed))
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
print("训练集", tuple(train_data.data.shape), "测试集", tuple(test_data.data.shape))

训练集 (60000, 28, 28) 测试集 (10000, 28, 28)


## 3. 创建 LeNet 风格模型

两层卷积采用 Sigmoid 和平均池化，后接三层全连接。输出为 10 个 logits，直接传给交叉熵。

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.clf = nn.Linear(84, 10)

    def forward(self, x):
        x = F.avg_pool2d(torch.sigmoid(self.conv1(x)), 2)
        x = F.avg_pool2d(torch.sigmoid(self.conv2(x)), 2)
        x = torch.flatten(x, 1)
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return self.clf(x)

model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(model)
print("参数量", sum(p.numel() for p in model.parameters()))

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (clf): Linear(in_features=84, out_features=10, bias=True)
)
参数量 61706


## 4. 定义训练与测试

损失按样本数加权，正确处理末批不足 512 的情况。测试时关闭梯度。混淆矩阵行是真实类别，列是预测类别。

In [4]:
def train_one_epoch():
    model.train()
    loss_sum, correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.numel()
        correct += (logits.argmax(1) == labels).sum().item()
    return loss_sum / len(train_data), correct / len(train_data)


@torch.inference_mode()
def evaluate():
    model.eval()
    loss_sum = 0.0
    matrix = torch.zeros(10, 10, dtype=torch.int64)
    for images, labels in test_loader:
        logits = model(images.to(device))
        loss_sum += F.cross_entropy(logits, labels.to(device), reduction="sum").item()
        predicted = logits.argmax(1).cpu()
        matrix += torch.bincount(10 * labels + predicted, minlength=100).reshape(10, 10)
    return loss_sum / len(test_data), matrix.trace().item() / len(test_data), matrix

## 5. 训练 30 轮并记录指标

每轮测试用于复现课程流程。最终报告使用第 30 轮模型，不根据测试准确率选择模型。

In [5]:
history = []
for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train_one_epoch()
    test_loss, test_accuracy, matrix = evaluate()
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "train_accuracy": train_accuracy, "test_loss": test_loss,
                    "test_accuracy": test_accuracy})
    print(f"{epoch:02}/{epochs} | train loss={train_loss:.4f} | "
          f"test loss={test_loss:.4f} | test accuracy={test_accuracy:.2%}")

01/30 | train loss=2.2968 | test loss=2.1403 | test accuracy=25.80%
02/30 | train loss=1.3813 | test loss=0.4709 | test accuracy=85.25%
03/30 | train loss=0.2642 | test loss=0.1760 | test accuracy=94.67%
04/30 | train loss=0.1459 | test loss=0.1319 | test accuracy=95.92%
05/30 | train loss=0.1046 | test loss=0.1018 | test accuracy=96.84%
06/30 | train loss=0.0836 | test loss=0.0743 | test accuracy=97.67%
07/30 | train loss=0.0739 | test loss=0.0678 | test accuracy=97.76%
08/30 | train loss=0.0650 | test loss=0.0597 | test accuracy=98.05%
09/30 | train loss=0.0536 | test loss=0.0578 | test accuracy=98.16%
10/30 | train loss=0.0522 | test loss=0.0495 | test accuracy=98.38%
11/30 | train loss=0.0465 | test loss=0.0545 | test accuracy=98.22%
12/30 | train loss=0.0409 | test loss=0.0521 | test accuracy=98.46%
13/30 | train loss=0.0392 | test loss=0.0488 | test accuracy=98.50%
14/30 | train loss=0.0340 | test loss=0.0496 | test accuracy=98.45%
15/30 | train loss=0.0354 | test loss=0.0484 | t

## 6. 绘制训练与测试曲线、混淆矩阵

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
steps = [row["epoch"] for row in history]
for split in ("train", "test"):
    axes[0].plot(steps, [row[f"{split}_loss"] for row in history], label=split)
    axes[1].plot(steps, [100 * row[f"{split}_accuracy"] for row in history], label=split)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
fig.savefig(run_dir / "training_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
plot = ax.imshow(matrix.numpy(), cmap="Blues")
for row in range(10):
    for column in range(10):
        count = matrix[row, column].item()
        color = "white" if count > matrix.max().item() / 2 else "black"
        ax.text(column, row, str(count), ha="center", va="center", color=color, fontsize=8)
ax.set(xlabel="Predicted digit", ylabel="True digit", xticks=range(10), yticks=range(10),
       title=f"MNIST confusion matrix ({test_accuracy:.2%})")
fig.colorbar(plot, ax=ax)
fig.savefig(run_dir / "confusion_matrix.png", dpi=150)
plt.show()

## 7. 查看同一张测试图像的中间特征

固定测试集第 0 张，保存输入图及 Conv1、Pool1、Conv2、Pool2 的全部通道。特征图共享 [0, 1] 灰度范围。

In [7]:
model.eval()
sample, label = test_data[0]
with torch.inference_mode():
    image = sample.unsqueeze(0).to(device)
    conv1 = torch.sigmoid(model.conv1(image))
    pool1 = F.avg_pool2d(conv1, 2)
    conv2 = torch.sigmoid(model.conv2(pool1))
    pool2 = F.avg_pool2d(conv2, 2)
    prediction = model(image).argmax(1).item()

fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")
ax.imshow(sample.squeeze(), cmap="gray", vmin=0, vmax=1)
ax.set_title(f"True {label}, predicted {prediction}")
ax.axis("off")
fig.savefig(run_dir / "feature_input.png", dpi=150)
plt.show()

features = {"conv1": conv1, "pool1": pool1, "conv2": conv2, "pool2": pool2}
for name, feature in features.items():
    maps = feature[0].cpu().numpy()
    rows = (len(maps) + 5) // 6
    fig, axes = plt.subplots(rows, 6, figsize=(10, rows * 1.8), squeeze=False,
                             layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for channel, (ax, feature_map) in enumerate(zip(axes.flat, maps)):
        ax.imshow(feature_map, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Channel {channel}", fontsize=9)
    fig.suptitle(f"{name}: {tuple(feature.shape)}")
    fig.savefig(run_dir / f"{name}_features.png", dpi=150)
    plt.show()

## 8. 保存指标与模型

metrics.json 包含环境、超参数、逐轮统计和最终混淆矩阵。模型权重保存在本次运行批次中。

In [8]:
metrics = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "device": str(device), "seed": seed, "epochs": epochs,
    "batch_size": batch_size, "learning_rate": learning_rate,
    "train_samples": len(train_data), "test_samples": len(test_data),
    "parameters": sum(p.numel() for p in model.parameters()),
    "history": history, "test_loss": test_loss, "test_accuracy": test_accuracy,
    "test_correct": matrix.trace().item(), "confusion_matrix": matrix.tolist(),
    "feature_sample": {"index": 0, "label": label, "prediction": prediction},
    "feature_shapes": {name: list(feature.shape) for name, feature in features.items()},
}
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
torch.save(model.cpu().state_dict(), run_dir / "lenet.pth")
print("结果保存至", run_dir)

结果保存至 5th.exp/result/20261119-134819-193069
